# 1.2 Train preprocessing

Second experimentation notebook of the house-price MLOps pipeline.

**Goal:** clean **train only**. The steps are row filters and type fixes, so they stay valid if `data_train.csv` gains duplicates or missing values later.

This notebook does **not** use test data, and it does **not** add columns. Encoding, scaling and new features belong in a later feature-engineering notebook.

| Artifact | Path |
|---|---|
| Input | `1-experimentation/data/data_train.csv` |
| Output | `1-experimentation/data/data_train_processed.csv` |


## 0. Setup

In [1]:
%pip install -q pandas



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

EXPECTED_COLUMNS = [
    "price",
    "sqft",
    "bedrooms",
    "bathrooms",
    "location",
    "year_built",
    "condition",
]
NUMERIC_COLUMNS = ["price", "sqft", "bedrooms", "bathrooms", "year_built"]
STRING_COLUMNS = ["location", "condition"]

TRAIN_PATH = "../data/data_train.csv"
TRAIN_PREPROCESSED_PATH = "../data/data_train_preprocessed.csv"


## 1. Load train

Only `data_train.csv`. Test stays untouched until a later notebook.


In [4]:
train_df = pd.read_csv(TRAIN_PATH)

print(f"Train: {train_df.shape[0]} rows x {train_df.shape[1]} cols")
print(f"Columns: {list(train_df.columns)}")
train_df.head()


Train: 67 rows x 7 cols
Columns: ['price', 'sqft', 'bedrooms', 'bathrooms', 'location', 'year_built', 'condition']


,price,sqft,bedrooms,bathrooms,location,year_built,condition
0,357000,1580,2,1.5,Suburb,1960,Fair
1,272000,1480,2,1.0,Rural,1952,Poor
2,752000,2526,3,2.5,Downtown,1998,Excellent
3,372000,1640,2,1.5,Suburb,1963,Fair
4,537000,2050,3,2.0,Urban,1990,Good


In [5]:
missing = [col for col in EXPECTED_COLUMNS if col not in train_df.columns]
extra = [col for col in train_df.columns if col not in EXPECTED_COLUMNS]
if missing:
    raise ValueError(f"Train is missing expected columns: {missing}")
if extra:
    print(f"Warning: unexpected columns will be dropped: {extra}")

train_df = train_df[EXPECTED_COLUMNS]
print("Schema matches expected columns.")


Schema matches expected columns.


## 2. Clean train

The current file may already be complete. These steps still run so a newer `data_train.csv` with duplicates or nulls is cleaned the same way.

1. Keep the original column set (no new features).
2. Strip whitespace in text columns; treat blank strings as missing.
3. Coerce numeric columns (invalid values become missing).
4. Drop duplicate rows.
5. Drop rows with any missing value.


In [6]:
def preprocess_train(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df[EXPECTED_COLUMNS].copy()
    log = []

    def record(step: str, before: int) -> None:
        log.append(
            {
                "step": step,
                "rows_before": before,
                "rows_after": len(out),
                "rows_removed": before - len(out),
            }
        )

    for col in STRING_COLUMNS:
        out[col] = out[col].astype("string").str.strip()
        out[col] = out[col].replace("", pd.NA)

    for col in NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    before = len(out)
    out = out.drop_duplicates()
    record("drop_duplicates", before)

    before = len(out)
    out = out.dropna()
    record("drop_missing_rows", before)

    out = out.reset_index(drop=True)
    return out, pd.DataFrame(log)


train_processed, cleaning_log = preprocess_train(train_df)
cleaning_log


,step,rows_before,rows_after,rows_removed
0,drop_duplicates,67,67,0
1,drop_missing_rows,67,67,0


### Sanity check

In [7]:
print(f"Rows in:  {len(train_df)}")
print(f"Rows out: {len(train_processed)}")
print(f"Columns in:  {list(train_df.columns)}")
print(f"Columns out: {list(train_processed.columns)}")
print(f"Same columns: {list(train_processed.columns) == EXPECTED_COLUMNS}")
print(f"Duplicates remaining: {train_processed.duplicated().sum()}")
print(f"Missing values remaining: {int(train_processed.isna().sum().sum())}")
train_processed.head()


Rows in:  67
Rows out: 67
Columns in:  ['price', 'sqft', 'bedrooms', 'bathrooms', 'location', 'year_built', 'condition']
Columns out: ['price', 'sqft', 'bedrooms', 'bathrooms', 'location', 'year_built', 'condition']
Same columns: True
Duplicates remaining: 0
Missing values remaining: 0


,price,sqft,bedrooms,bathrooms,location,year_built,condition
0,357000,1580,2,1.5,Suburb,1960,Fair
1,272000,1480,2,1.0,Rural,1952,Poor
2,752000,2526,3,2.5,Downtown,1998,Excellent
3,372000,1640,2,1.5,Suburb,1963,Fair
4,537000,2050,3,2.0,Urban,1990,Good


In [8]:
pd.DataFrame({
    "dtype": train_processed.dtypes.astype(str),
    "non_null": train_processed.notna().sum(),
    "nulls": train_processed.isna().sum(),
    "n_unique": train_processed.nunique(),
})


,dtype,non_null,nulls,n_unique
price,int64,67,0,66
sqft,int64,67,0,55
bedrooms,int64,67,0,4
bathrooms,float64,67,0,7
location,string,67,0,6
year_built,int64,67,0,47
condition,string,67,0,4


## 3. Persist cleaned train

In [12]:
train_processed.to_csv(TRAIN_PREPROCESSED_PATH, index=False)
print(f"Wrote {TRAIN_PREPROCESSED_PATH} ({len(train_processed)} rows, {train_processed.shape[1]} cols)")

reloaded = pd.read_csv(TRAIN_PREPROCESSED_PATH)
assert list(reloaded.columns) == EXPECTED_COLUMNS
assert len(reloaded) == len(train_processed)
print("Reload check passed.")


Wrote ../data/data_train_preprocessed.csv (67 rows, 7 cols)
Reload check passed.


## 4. Findings for later notebooks

- Preprocessing is train-only and column-preserving: same 7 columns as the raw train file.
- Duplicates and missing rows are dropped, so the notebook still works if `data_train.csv` changes.
